In [111]:
import numpy as np
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB

In [112]:
dfTrue = pd.read_csv('True.csv')
dfFake = pd.read_csv('Fake.csv')
dfTrue['NewsType'] = 1
dfFake['NewsType'] = 0

In [113]:
data = pd.concat([dfTrue,dfFake])
data = data.drop_duplicates(subset=['text'])
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
data['original_text'] = data['title'] + " " + data['text']
data = data[['original_text', 'NewsType']]

In [114]:
def remove_tags(text):
    text = str(text)
    clean_text = re.sub(re.compile('<.*?>'),'',text)
    return clean_text

def remove_urls(text):
    text = str(text)
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'',text)

exclude = string.punctuation
def remove_punc(text):
    return text.translate(str.maketrans('','',exclude))
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])
def remove_publisher_dateline(text):
    text = str(text)
    clean_text = re.sub(r'^.*?\(Reuters\)\s*-\s*', '', text, flags=re.IGNORECASE)
    clean_text = re.sub(r'\breuters\b', '', clean_text, flags=re.IGNORECASE)
    return clean_text
data['original_text']= data['original_text'].apply(remove_tags)
data['original_text'] = data['original_text'].apply(lambda x: x.lower())
data['original_text'] = data['original_text'].apply(remove_urls)
data['original_text'] = data['original_text'].apply(remove_publisher_dateline) # N
data['original_text'] = data['original_text'].apply(remove_punc)
data['original_text'] = data['original_text'].apply(remove_stopwords)

In [115]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    data['original_text'],
    data['NewsType'],
    test_size=0.2,
    random_state=42,
    stratify=data['NewsType']
)
tfidf = TfidfVectorizer(max_features=5000)

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)
print("Text representation complete! Vectorized matrix shape:", X_train.shape)

Text representation complete! Vectorized matrix shape: (30916, 5000)


In [116]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Passive Aggressive Classifier": PassiveAggressiveClassifier(max_iter=50, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}
results = []
for model_name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    # Predict
    y_pred = model.predict(X_test)
    # Evaluate
    accuracy = accuracy_score(y_test, y_pred)
    results.append({"Architecture": model_name, "Accuracy (%)": round(accuracy * 100, 2)})

comparison_df = pd.DataFrame(results).sort_values(by="Accuracy (%)", ascending=False).reset_index(drop=True)

print("📊 Model Comparison Results:")
print(comparison_df)

def predict_custom_news(custom_text):
    cleaned_text = remove_tags(custom_text)
    cleaned_text = cleaned_text.lower() 
    cleaned_text = remove_urls(cleaned_text)
    cleaned_text = remove_punc(cleaned_text)
    cleaned_text = remove_stopwords(cleaned_text)
    

    vectorized_text = tfidf.transform([cleaned_text])

    prediction = rf.predict(vectorized_text)
    probabilities = rf.predict_proba(vectorized_text)
    
    fake_prob = probabilities[0][0] * 100
    real_prob = probabilities[0][1] * 100
    
    print(f"Text: '{custom_text}'\n")
    print(f"Probability Breakdowns:")
    print(f"Fake News Confidence: {fake_prob:.2f}%")
    print(f" Real News Confidence: {real_prob:.2f}%")
    print("-" * 40)
    if real_prob > fake_prob:
        print("Verdict: REAL NEWS")
    else:
        print("Verdict: FAKE NEWS")

my_news = "BREAKING: Government insiders reveal that aliens have completely taken over the Pentagon! The president has secretly surrendered to the Martian fleet, and mandatory microchips will be implanted in all citizens by tomorrow morning. WAKE UP!"
predict_custom_news(my_news)

/Users/ravish/NumPy/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)


📊 Model Comparison Results:
                    Architecture  Accuracy (%)
0  Passive Aggressive Classifier         98.58
1            Logistic Regression         98.28
2                  Random Forest         98.20
3        Multinomial Naive Bayes         93.54
Text: 'BREAKING: Government insiders reveal that aliens have completely taken over the Pentagon! The president has secretly surrendered to the Martian fleet, and mandatory microchips will be implanted in all citizens by tomorrow morning. WAKE UP!'

Probability Breakdowns:
Fake News Confidence: 63.00%
 Real News Confidence: 37.00%
----------------------------------------
Verdict: FAKE NEWS
